# HallusionBench — Google Colab Runner theo từng Mô hình (A → B → C + Gói Kết quả chuẩn Repo & MMHal)

**Zero-shot evaluation (0 epoch — KHÔNG train/fine-tune).**
- **Cell 1:** Mount Google Drive + kiểm tra GPU / VRAM / BF16 native + **liệt kê sẵn các mô hình đã chạy trên Drive** (để biết thiếu model nào).
- **Cell 2:** Clone/pull nhánh `benchmark/hallusionbench` + cài thư viện chuẩn cho **SmolVLM2** & **Qwen3-VL** (riêng **Phi-4-multimodal** được **Cell 5C** tự tạo venv tách biệt `/content/venv_phi4` với `transformers==4.48.2` nên bấm **Run all** 1 phát chạy đủ cả 3 model không cần Restart!).
- **Cell 3:** Chạy kiểm tra cấu trúc repo & toàn bộ unit/E2E tests (`test_hallusionbench.py`).
- **Cell 4 (Phần A):** Nạp dataset `rayguan/HallusionBench` từ Drive cache (hoặc tải 1 lần), chạy `audit` & tạo `pilot_ids_8sub.json` phủ đủ **8/8 dạng bài**.
- **Cell 5 (Cấu hình chung) + Cell 5A / 5B / 5C (CHIA RIÊNG TỪNG MODEL):** Chạy độc lập từng mô hình (`smolvlm2`, `qwen3_vl`, `phi4_multimodal`), tự lưu từng mẫu vào Drive & tự resume nếu ngắt.
- **Cell 6 (Phần C):** Tự phát hiện mọi mô hình đã chạy trên Drive, chấm `aAcc / fAcc / qAcc`, tự bổ sung bộ file dễ đọc giống MMHal (`*_responses.json`, `*_evaluation.json`, `*_summary.json`, `inference_config.json`, `inference_summary.json`) và kiểm chứng `check_results.py` & `check_manifest.py`.
- **Cell 7 (Đóng gói chuẩn Git):** Xuất thẳng cấu trúc `benchmarks/hallusionbench/results/<model>/` (đủ **7 artifact chuẩn + bộ file MMHal + README bảng điểm + `analysis/error_cases.md` + `cross_model_disagreements.json`**).
- **Cell 8:** Nén 1 file `.zip` đúng cấu trúc repo để giải nén thẳng vào thư mục gốc repo rồi `git add` & `git push`.

## Cell 1 — Kết nối Google Drive, kiểm tra GPU & xem các run đã có trên Drive

In [ ]:
import json
import os
import sys
from pathlib import Path
import torch
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/hallusionbench_runs")
OUTPUT_DIR = DRIVE_ROOT / "results"
AUDIT_DIR = DRIVE_ROOT / "audit_and_pilot"
DELIVERABLES_DIR = DRIVE_ROOT / "repo_deliverables"
DRIVE_DATASET_CACHE = DRIVE_ROOT / "dataset_cache"
LOCAL_DATA_ROOT = Path("/content/data/HallusionBench")
LOCAL_HF_CACHE = Path("/content/hf_cache")

for p in (OUTPUT_DIR, AUDIT_DIR, DELIVERABLES_DIR, DRIVE_DATASET_CACHE, LOCAL_DATA_ROOT.parent, LOCAL_HF_CACHE):
    p.mkdir(parents=True, exist_ok=True)

os.environ["DATA_ROOT"] = str(LOCAL_DATA_ROOT)
os.environ["HF_HOME"] = str(LOCAL_HF_CACHE)
os.environ["MODEL_CACHE_DIR"] = str(LOCAL_HF_CACHE)
os.environ["OUTPUT_DIR"] = str(OUTPUT_DIR)
os.environ["AUDIT_DIR"] = str(AUDIT_DIR)
os.environ["DELIVERABLES_DIR"] = str(DELIVERABLES_DIR)
os.environ["DRIVE_DATASET_CACHE"] = str(DRIVE_DATASET_CACHE)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Python:", sys.version)
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
bf16_emulated = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
try:
    bf16_native = torch.cuda.is_bf16_supported(including_emulation=False) if torch.cuda.is_available() else False
except TypeError:
    bf16_native = bf16_emulated and (torch.cuda.get_device_properties(0).major >= 8)
print(f"GPU count: {torch.cuda.device_count()} | BF16 native: {bf16_native} | BF16 emulated: {bf16_emulated}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | sm_{props.major}{props.minor} | {round(props.total_memory / 2**30, 2)} GiB")
!nvidia-smi

print("\n=== CÁC RUN ĐÃ LƯU TRÊN GOOGLE DRIVE ===")
found_runs = sorted(OUTPUT_DIR.rglob("run_status.json"))
if not found_runs:
    print("(Chưa có run nào trong", OUTPUT_DIR, ")")
else:
    for st_file in found_runs:
        st = json.loads(st_file.read_text(encoding="utf-8"))
        rel = st_file.parent.relative_to(OUTPUT_DIR)
        print(f" - {rel}: status={st.get('status')} | completed={st.get('completed_samples')}/{st.get('total_samples')} | failed={st.get('failed_samples', 0)}")

## Cell 2 — Clone/pull nhánh `benchmark/hallusionbench` & cài thư viện (SmolVLM2 & Qwen3-VL)

*(Không cần đổi biến môi trường khi bấm **Run all**: Cell 2 cài môi trường cho SmolVLM2 & Qwen3-VL, còn **Cell 5C** sẽ tự tạo venv riêng `/content/venv_phi4` với `transformers==4.48.2` cho Phi-4-multimodal.)*

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/hoainhan256/vlm-hallucination-benchmark.git"
BRANCH = "benchmark/hallusionbench"
REPO_DIR = Path("/content/vlm-hallucination-benchmark")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

%cd /content/vlm-hallucination-benchmark
!git rev-parse HEAD
!ls -la benchmarks/hallusionbench

missing_core = [p for p in ("vlm_benchmark", "benchmarks/hallusionbench/dataset.py", "benchmarks/hallusionbench/inference.py", "benchmarks/hallusionbench/evaluation.py") if not Path(p).exists()]
if missing_core:
    raise RuntimeError(f"Nhánh {BRANCH} thiếu file cốt lõi: {missing_core}")

!pip install -q -r requirements.txt
!pip install -q --upgrade transformers accelerate qwen-vl-utils num2words

import transformers, accelerate
print("Môi trường chính sẵn sàng (SmolVLM2 & Qwen3-VL) -> transformers:", transformers.__version__, "| accelerate:", accelerate.__version__)

## Cell 3 — Kiểm tra cấu trúc repo & chạy toàn bộ unit tests (A + B + C + E2E)

In [ ]:
import sys
from pathlib import Path

repo_root = str(Path("/content/vlm-hallucination-benchmark").resolve())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from vlm_benchmark.config import validate_all

print("Trạng thái khóa configs chung (official gate):")
for item in validate_all("configs/models.yaml", "configs/datasets.yaml", "configs/experiment.yaml"):
    print(" -", item)

!python scripts/check_repository.py
!python -m pytest -v tests/ benchmarks/hallusionbench/tests/

## Cell 4 (PHẦN A) — Nạp dataset từ Drive (hoặc tải 1 lần lưu lên Drive), Audit & xuất Pilot 8/8 dạng bài

In [ ]:
import json
import os
import shutil
import tarfile
from pathlib import Path
from huggingface_hub import snapshot_download

HF_DATASET_REPO = "rayguan/HallusionBench"
HF_DATASET_REVISION = "2067dfdce5abc7efdbb0927f4721ae282f0b6c51"
LOCAL_DATA_ROOT = Path(os.environ["DATA_ROOT"])
AUDIT_DIR = Path(os.environ["AUDIT_DIR"])
DRIVE_DATASET_TAR = Path(os.environ["DRIVE_DATASET_CACHE"]) / f"HallusionBench_{HF_DATASET_REVISION[:8]}.tar"
AUDIT_REPORT = AUDIT_DIR / "hallusionbench_audit_report.json"
PILOT_FILE = AUDIT_DIR / "pilot_ids_8sub.json"

if not (LOCAL_DATA_ROOT / "HallusionBench.json").is_file():
    if DRIVE_DATASET_TAR.is_file():
        print(f"Nạp dataset lưu sẵn từ Google Drive: {DRIVE_DATASET_TAR}...")
        if LOCAL_DATA_ROOT.exists():
            shutil.rmtree(LOCAL_DATA_ROOT)
        LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
        with tarfile.open(DRIVE_DATASET_TAR, "r") as tf:
            tf.extractall(LOCAL_DATA_ROOT)
        print("Giải nén xong!")
    else:
        print(f"Lần đầu: tải {HF_DATASET_REPO} @ {HF_DATASET_REVISION}...")
        snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset", revision=HF_DATASET_REVISION, local_dir=str(LOCAL_DATA_ROOT))
        tmp_tar = Path("/content/HallusionBench_tmp.tar")
        with tarfile.open(tmp_tar, "w") as tf:
            for item in LOCAL_DATA_ROOT.iterdir():
                if item.name != ".cache":
                    tf.add(item, arcname=item.name)
        shutil.move(str(tmp_tar), str(DRIVE_DATASET_TAR))
        print("Đã lưu vĩnh viễn trên Drive:", DRIVE_DATASET_TAR)

os.environ["DATASET_REV"] = HF_DATASET_REVISION
os.environ["PILOT_FILE"] = str(PILOT_FILE)

!python -m benchmarks.hallusionbench.dataset \
  --mode audit \
  --dataset-json "$DATA_ROOT/HallusionBench.json" \
  --data-root "$DATA_ROOT" \
  --dataset-source "rayguan/HallusionBench" \
  --dataset-revision "$DATASET_REV" \
  --output "$AUDIT_DIR/hallusionbench_audit_report.json"

!python -m benchmarks.hallusionbench.dataset \
  --mode export-pilot \
  --dataset-json "$DATA_ROOT/HallusionBench.json" \
  --data-root "$DATA_ROOT" \
  --dataset-source "rayguan/HallusionBench" \
  --dataset-revision "$DATASET_REV" \
  --target-groups 8 \
  --output "$AUDIT_DIR/pilot_ids_8sub.json"

!python -m benchmarks.hallusionbench.dataset \
  --mode verify-pilot \
  --dataset-json "$DATA_ROOT/HallusionBench.json" \
  --data-root "$DATA_ROOT" \
  --pilot-file "$AUDIT_DIR/pilot_ids_8sub.json"

pilot = json.loads(PILOT_FILE.read_text(encoding="utf-8"))
print(f"\nPilot 8 dạng bài: {pilot['sample_count']} mẫu ({pilot['group_count']} nhóm) | Subcategories: {pilot['breakdown_by_subcategory']}")

## Cell 5 (PHẦN B — Cấu hình chung) — Chọn `STAGE` & hàm chạy riêng từng mô hình

- `STAGE = "pilot"`: Chạy tập **8/8 subcategory** (`pilot_ids_8sub.json`).
- `STAGE = "full_baseline"`: Chạy **TOÀN BỘ 1.129 câu / 346 ảnh** (tự lưu từng mẫu lên Drive & tự resume).
- `STAGE = "smoke_test"`: Chạy nhanh 2 câu kiểm tra kỹ thuật.

Sau khi chạy Cell 5 này, bạn chỉ cần chạy **Cell 5A**, **Cell 5B**, hoặc **Cell 5C** cho đúng mô hình cần chạy/còn thiếu!

In [ ]:
import os
import subprocess
import torch
from pathlib import Path

STAGE = "pilot"          # "pilot" (8/8 dạng bài) | "full_baseline" (1.129 câu) | "smoke_test" (2 câu)
MAX_NEW_TOKENS = 128
MAX_SAMPLES = 2          # Chỉ dùng khi STAGE == "smoke_test"

try:
    bf16_native = torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    bf16_native = torch.cuda.is_available() and torch.cuda.is_bf16_supported() and (torch.cuda.get_device_properties(0).major >= 8)

DTYPE = "bfloat16" if bf16_native else "float16"
gpu_tag = "bf16gpu" if bf16_native else "t4"
EXPERIMENT_ID = f"hb_colab_{gpu_tag}_{STAGE}"
os.environ["EXPERIMENT_ID"] = EXPERIMENT_ID
os.environ["STAGE"] = STAGE
os.environ["DTYPE"] = DTYPE

RUN_DIRS = {
    m: f"{os.environ['OUTPUT_DIR']}/{EXPERIMENT_ID}/{STAGE}_{m}_{DTYPE}"
    for m in ("smolvlm2", "qwen3_vl", "phi4_multimodal")
}

def run_single_model(
    model_key: str,
    *,
    python_exe: str = "python",
    attn_impl: str | None = None,
    max_gpu_memory: str | None = None,
    max_cpu_memory: str | None = None,
    offload_folder: str | None = None,
) -> int:
    if attn_impl is None:
        attn_impl = "eager" if (model_key == "phi4_multimodal" and not bf16_native) else "sdpa"
    run_id = f"{STAGE}_{model_key}_{DTYPE}"
    run_dir = RUN_DIRS[model_key]

    cmd = [
        python_exe, "-m", "benchmarks.hallusionbench.inference",
        "--mode", "run",
        "--allow-model-load",
        "--model-key", model_key,
        "--stage", STAGE,
        "--dtype", DTYPE,
        "--attn-implementation", attn_impl,
        "--max-new-tokens", str(MAX_NEW_TOKENS),
        "--dataset-json", f"{os.environ['DATA_ROOT']}/HallusionBench.json",
        "--data-root", os.environ["DATA_ROOT"],
        "--dataset-source", "rayguan/HallusionBench",
        "--dataset-revision", os.environ["DATASET_REV"],
        "--experiment-id", EXPERIMENT_ID,
        "--run-id", run_id,
        "--output-root", os.environ["OUTPUT_DIR"],
    ]
    if model_key == "phi4_multimodal":
        cmd.append("--trust-remote-code")
    if max_gpu_memory:
        cmd.extend(["--max-gpu-memory", max_gpu_memory])
    if max_cpu_memory:
        cmd.extend(["--max-cpu-memory", max_cpu_memory])
    if offload_folder:
        cmd.extend(["--offload-folder", offload_folder])
    if STAGE == "smoke_test":
        cmd.extend(["--max-samples", str(MAX_SAMPLES)])
    elif STAGE == "pilot":
        cmd.extend(["--pilot-file", os.environ["PILOT_FILE"]])

    print("\n" + "=" * 72)
    print(f"MODEL: {model_key} | PY: {python_exe} | STAGE: {STAGE} | DTYPE: {DTYPE} | ATTN: {attn_impl} | OFFLOAD: {max_gpu_memory}")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    print(f"--> Xong {model_key} (exit={rc}) | Output: {run_dir}")
    if rc != 0:
        st_path = Path(run_dir) / "run_status.json"
        if st_path.is_file():
            print("LÝ DO LỖI/BLOCKED:", st_path.read_text(encoding="utf-8"))
    return rc

print(f"Đã sẵn sàng EXPERIMENT_ID = {EXPERIMENT_ID} (DTYPE={DTYPE})")
for k, v in RUN_DIRS.items():
    exists = (Path(v) / "predictions.jsonl").is_file()
    print(f" - {k:16s}: {'ĐÃ CÓ FILE' if exists else 'CHƯA CHẠY'} -> {v}")

### Cell 5A — Chạy riêng Model 1: `smolvlm2` (`HuggingFaceTB/SmolVLM2-2.2B-Instruct`)

In [ ]:
run_single_model("smolvlm2")

### Cell 5B — Chạy riêng Model 2: `qwen3_vl` (`Qwen/Qwen3-VL-4B-Instruct`)

In [ ]:
run_single_model("qwen3_vl")

### Cell 5C — Chạy riêng Model 3: `phi4_multimodal` (`microsoft/Phi-4-multimodal-instruct`)
- **Tự tạo venv tách biệt `/content/venv_phi4`** (kế thừa PyTorch/CUDA có sẵn trên Colab, cài riêng `transformers==4.48.2`, `accelerate==1.3.0`, `peft==0.13.2`, `huggingface-hub<1.0`, `backoff`, `soundfile`, `scipy`) để **bấm Run all chạy đủ cả 3 model một mạch không cần Restart session** và không đụng môi trường của SmolVLM2/Qwen3-VL.
- Trên GPU T4 15GB, tự bật CPU offload (`--max-gpu-memory 8GiB --max-cpu-memory 10GiB --offload-folder /content/phi4_offload`) đúng chính sách `docs/model_policy.md` (không sửa ảnh, không đổi dtype, không lượng tử hóa) để tránh OOM.

In [ ]:
import subprocess
import sys
from pathlib import Path

PHI4_VENV = Path("/content/venv_phi4")
PHI4_PY = PHI4_VENV / "bin" / "python"
PHI4_PIP = PHI4_VENV / "bin" / "pip"

def _phi4_venv_ready() -> bool:
    if not PHI4_PY.is_file():
        return False
    chk = subprocess.run(
        [
            str(PHI4_PY), "-c",
            "import torch, transformers, accelerate, peft, backoff, soundfile; "
            "assert transformers.__version__ == '4.48.2'",
        ],
        capture_output=True,
        text=True,
    )
    return chk.returncode == 0

if not _phi4_venv_ready():
    print("Tạo/cài môi trường venv tách biệt cho Phi-4-multimodal (/content/venv_phi4)...")
    if not PHI4_PIP.is_file():
        res_venv = subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(PHI4_VENV)])
        if res_venv.returncode != 0 or not PHI4_PIP.is_file():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
            subprocess.run([sys.executable, "-m", "virtualenv", "--system-site-packages", str(PHI4_VENV)], check=True)
    subprocess.run(
        [
            str(PHI4_PIP), "install", "-q",
            "transformers==4.48.2",
            "accelerate==1.3.0",
            "peft==0.13.2",
            "huggingface-hub>=0.24.0,<1.0",
            "tokenizers>=0.21,<0.22",
            "backoff", "soundfile", "scipy", "pyyaml", "einops",
        ],
        check=True,
    )

subprocess.run(
    [
        str(PHI4_PY), "-c",
        "import torch, transformers, accelerate, peft, huggingface_hub, backoff, soundfile; "
        "print('Phi-4 venv OK -> torch:', torch.__version__, '| transformers:', transformers.__version__, "
        "'| accelerate:', accelerate.__version__, '| peft:', peft.__version__, '| hf_hub:', huggingface_hub.__version__)",
    ],
    check=True,
)

offload_kwargs = {}
if not bf16_native:
    offload_kwargs = {
        "attn_impl": "eager",
        "max_gpu_memory": "8GiB",
        "max_cpu_memory": "10GiB",
        "offload_folder": "/content/phi4_offload",
    }
run_single_model("phi4_multimodal", python_exe=str(PHI4_PY), **offload_kwargs)

## Cell 6 (PHẦN C) — Tự phát hiện mọi model đã có trên Drive, chấm điểm, bổ sung file MMHal & kiểm chứng 7 artifact

In [ ]:
import json
import os
import subprocess
from pathlib import Path

AVAILABLE_RUNS = {}
summary_rows = []
for model_key, run_dir_str in RUN_DIRS.items():
    run_dir = Path(run_dir_str)
    if not (run_dir / "predictions.jsonl").is_file() or not (run_dir / "run_status.json").is_file():
        print(f"[BỎ QUA] {model_key}: chưa có run tại {run_dir}")
        continue

    status_data = json.loads((run_dir / "run_status.json").read_text(encoding="utf-8"))
    meta_data = json.loads((run_dir / "run_metadata.json").read_text(encoding="utf-8"))

    if status_data.get("status") == "BLOCKED":
        print(f"[BLOCKED] {model_key}: {status_data.get('reasons')}")
    else:
        res = subprocess.run(
            [
                "python", "-m", "benchmarks.hallusionbench.evaluation",
                "--mode", "score",
                "--run-dir", str(run_dir),
                "--dataset-json", f"{os.environ['DATA_ROOT']}/HallusionBench.json",
                "--data-root", os.environ["DATA_ROOT"],
            ],
            capture_output=True,
            text=True,
        )
        if res.returncode != 0:
            print(res.stdout)
            print(res.stderr)
            res.check_returncode()
        AVAILABLE_RUNS[model_key] = str(run_dir)

    for chk_cmd in (
        ["python", "scripts/check_results.py", str(run_dir / "predictions.jsonl")],
        ["python", "scripts/check_manifest.py", str(run_dir)],
    ):
        chk = subprocess.run(chk_cmd, capture_output=True, text=True)
        if chk.returncode != 0:
            print(chk.stdout)
            print(chk.stderr)
            chk.check_returncode()

    metrics_data = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
    m = metrics_data.get("metrics") or {}
    c = metrics_data.get("summary_counts") or {}
    summary_rows.append({
        "model": model_key,
        "status": status_data.get("status"),
        "dtype": meta_data.get("dtype"),
        "model_rev": str(meta_data.get("resolved_model_revision"))[:10],
        "scored_q": c.get("scored_questions", 0),
        "aAcc": m.get("aAcc"),
        "fAcc": m.get("fAcc"),
        "qAcc": m.get("qAcc"),
        "errors": c.get("error_records", 0),
    })

print(f"\n=== BẢNG TỔNG HỢP CÁC MÔ HÌNH ĐÃ CHẠY (STAGE: {os.environ['STAGE']}) ===")
print("| Model | Status | Dtype | Scored Q | aAcc | fAcc | qAcc | Errors | Model SHA |")
print("|---|---|---|---:|---:|---:|---:|---:|---|")
for r in summary_rows:
    print(f"| `{r['model']}` | `{r['status']}` | `{r['dtype']}` | {r['scored_q']} | {r['aAcc']} | {r['fAcc']} | {r['qAcc']} | {r['errors']} | `{r['model_rev']}` |")

## Cell 7 (PHẦN C) — Đóng gói chuẩn repo (`benchmarks/hallusionbench/results/`) kết hợp 7 artifact + format MMHal

In [ ]:
import os
import subprocess
from pathlib import Path

if not AVAILABLE_RUNS:
    raise RuntimeError("Chưa có mô hình nào hoàn tất trong AVAILABLE_RUNS để đóng gói!")

REPO_RESULTS_OUT = Path(os.environ["DELIVERABLES_DIR"]) / os.environ["EXPERIMENT_ID"] / "benchmarks" / "hallusionbench" / "results"
run_dir_args = [f"{m}={p}" for m, p in AVAILABLE_RUNS.items()]

res = subprocess.run(
    [
        "python", "-m", "benchmarks.hallusionbench.evaluation",
        "--mode", "export-deliverables",
        "--overwrite-export",
        "--run-dirs", *run_dir_args,
        "--dataset-json", f"{os.environ['DATA_ROOT']}/HallusionBench.json",
        "--data-root", os.environ["DATA_ROOT"],
        "--output-dir", str(REPO_RESULTS_OUT),
    ],
    capture_output=True,
    text=True,
)
if res.stdout:
    print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    res.check_returncode()

# Kiểm chứng lại toàn bộ thư mục xuất bằng script kiểm tra dùng chung của repo
for m in AVAILABLE_RUNS:
    m_dir = REPO_RESULTS_OUT / m
    for chk_cmd in (
        ["python", "scripts/check_results.py", str(m_dir / "predictions.jsonl")],
        ["python", "scripts/check_manifest.py", str(m_dir)],
    ):
        chk = subprocess.run(chk_cmd, capture_output=True, text=True)
        print(chk.stdout.strip())
        if chk.returncode != 0:
            print(chk.stderr)
            chk.check_returncode()

print("\n=== CẤU TRÚC GÓI KẾT QUẢ SẴN SÀNG COMMIT ===")
for p in sorted(REPO_RESULTS_OUT.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(REPO_RESULTS_OUT.parents[2]))

## Cell 8 — Nén 1 file `.zip` đúng cấu trúc repo & tải về máy để commit lên nhánh `benchmark/hallusionbench`

In [ ]:
import os
import zipfile
from pathlib import Path
from google.colab import files

exp_id = os.environ["EXPERIMENT_ID"]
staging_root = Path(os.environ["DELIVERABLES_DIR"]) / exp_id
bundle_zip = Path(os.environ["OUTPUT_DIR"]) / f"hallusionbench_repo_results_{exp_id}.zip"

with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(staging_root.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=str(path.relative_to(staging_root)))

print(f"Đã đóng gói ({round(bundle_zip.stat().st_size / 1024, 1)} KB): {bundle_zip}")
print("Hướng dẫn: Giải nén trực tiếp vào thư mục gốc repo -> sẽ tự đổ vào benchmarks/hallusionbench/results/ -> git add & push!")
files.download(str(bundle_zip))